# 03 — Fine-tune ba Transformer và demo C/D
**Quốc Khánh: BERT cased · Đức Trí: RoBERTa · Nhật Huy: DistilBERT + demo.**

Notebook giải thích code và đọc artifacts, **không tự tải model/train/test**. Chạy khi mới có 1/9 run vẫn hiển thị từng seed đã hoàn tất và trạng thái các seed còn lại. Chỉ tính mean±std khi cả ba kiến trúc đủ ít nhất ba seed.

Run incomplete được ghi rõ; JSON sai, hash sai hoặc artifacts complete bị thiếu sẽ báo lỗi, không che bằng “đang chạy”. Mở kernel của môi trường đã cài extra Transformer; xem `docs/GETTING_STARTED.md`. [Bài GoEmotions](https://aclanthology.org/2020.acl-main.372/).

In [ ]:
from pathlib import Path
import json
import sys
import math
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Mở notebook từ repo hoặc thư mục notebooks")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.datasets.goemotions import REVISION
from src.models.transformer import (ARCHITECTURES, training_config, run_folder, load_transformer_run,
                        summarize_architectures)
print("Repo:", ROOT)
print("Kernel Python:", sys.executable)
labels = json.loads((ROOT / "data/labels.json").read_text(encoding="utf-8"))
print("28 nhãn:", labels)

## 1. Vì sao đây là đa nhãn?
Một câu có thể vừa vui (`joy`) vừa cảm ơn (`gratitude`). Nhãn thật là vector multi-hot 28 cột, ví dụ hai vị trí trên đều bằng 1.

**Văn bản → tokenizer riêng → encoder pretrained → 28 logits → sigmoid → ngưỡng độc lập.**
Huấn luyện dùng **BCEWithLogitsLoss** với nhãn float; không dùng softmax/argmax.
`neutral` là nhãn trong dataset, không tự gán khi model không chọn được nhãn.

Phần A của Duy dùng TF-IDF + Logistic Regression để làm mốc so sánh. C học biểu diễn ngữ cảnh bằng fine-tuning.

In [ ]:
# Cấu hình thật được khai báo trong helper; không chép số vào notebook riêng.
for architecture in ARCHITECTURES:
    print(architecture, training_config(architecture))

## 2. Chạy thử nhỏ trước
Lệnh dưới dùng 64 train, 32 validation, 1 epoch. Chỉ là kiểm luồng. **Không dùng số smoke cho bảng báo cáo cuối.**

Cài PyTorch phù hợp GPU/CPU, rồi requirements-transformer.txt. Xem docs/EXPERIMENTS.md.

In [ ]:
print("python -m scripts.transformers.train_transformer --architecture bert --seed 42 --smoke --device auto")
# Chạy dòng in trên ở terminal từ ROOT khi đã sẵn sàng tải model.

## 3. Thực nghiệm chính
Mỗi kiến trúc có ít nhất ba seed; nhóm dùng **42, 123, 2026**. Cấu hình hiện tại là **batch 16 / accumulation 1 / effective batch 16**, tối đa 128 token, `padding=dynamic_batch_trim`.
Dataset cache 128 token; collate chỉ bỏ padding bên phải sau câu dài nhất trong batch, không cắt token hợp lệ hoặc nhãn N×28.

Chọn epoch bằng validation Macro-F1 @0,5; hòa giữ epoch sớm. C1 BERT cased tham khảo lr 5e-5/4 epoch/batch hiệu dụng 16 của bài nền tảng; AdamW/warmup/max tokens/AMP/padding và C2/C3 là lựa chọn có khai báo của nhóm.

`--resume` bỏ qua run **đã complete**, đúng config/hash; không khôi phục optimizer giữa epoch. Run dở chạy lại riêng với `--overwrite`. Dùng `scripts.pipeline.complete_project --restart-incomplete` để điều phối, giữ các run complete. Không mở test để chọn cấu hình.

In [ ]:
for architecture in ARCHITECTURES:
    print(ARCHITECTURES[architecture]["owner"], "→",
          f"python -m scripts.transformers.run_transformer_seeds --architectures {architecture} "
          "--seeds 42 123 2026 --device cuda --resume")
print("Chạy tiếp toàn pipeline và chạy lại run dở:")
print("python -m scripts.pipeline.complete_project --device cuda --restart-incomplete")

## 4. Trạng thái và kết quả từng seed
Ô dưới chỉ đọc files. Run chưa tạo → “chưa có”; run `completed=false` → trạng thái incomplete và số epoch log tạm. Run `completed=true` → kiểm hash/checkpoint/mapping rồi hiển thị F1 full validation.

Cột F1 cuối chỉ lấy run complete. F1 của epoch trong run dở là thông tin tiến độ, không thay bảng thực nghiệm. Nếu artifacts bị hỏng, ô báo lỗi để kiểm nguyên nhân; không bắt mọi exception rồi giả là chưa xong.

In [ ]:
def read_c_status(root, label_names):
    progress, final_rows = [], []
    full_runs = {architecture: [] for architecture in ARCHITECTURES}
    for architecture in ARCHITECTURES:
        for seed in (42, 123, 2026):
            folder = run_folder(root, architecture, seed)
            metadata_path = folder / "run_metadata.json"
            row = {"architecture": architecture, "seed": seed, "status": "chưa có run",
                   "logged_epochs": 0, "planned_epochs": training_config(architecture)["epochs"],
                   "last_epoch_macro_f1_progress_only": None}
            if not metadata_path.exists():
                if folder.exists() and any(folder.iterdir()):
                    raise FileNotFoundError(f"Folder có artifact nhưng thiếu metadata: {folder}")
                progress.append(row)
                continue
            saved = json.loads(metadata_path.read_text(encoding="utf-8"))
            if (saved.get("artifact_version") != 1 or saved.get("method") != "C" or saved.get("architecture") != architecture
                    or saved.get("seed") != seed or saved.get("smoke") is not False
                    or saved.get("data_revision") != REVISION or saved.get("label_names") != label_names
                    or not isinstance(saved.get("completed"), bool)):
                raise ValueError(f"Metadata seed/kiến trúc/nhãn/full sai: {folder}")
            history = saved["history"]
            if (not isinstance(history, list) or len(history) > saved["config"]["epochs"]
                    or [entry["epoch"] for entry in history] != list(range(1, len(history) + 1))
                    or any(not math.isfinite(entry["macro_f1"]) or not 0 <= entry["macro_f1"] <= 1
                           for entry in history)):
                raise ValueError(f"History epoch/F1 không hợp lệ: {folder}")
            row.update(batch_size=saved["config"]["batch_size"],
                       gradient_accumulation=saved["config"]["gradient_accumulation"],
                       padding=saved["config"].get("padding", "cấu hình cũ: chưa ghi padding"))
            row.update(logged_epochs=len(history), planned_epochs=saved["config"]["epochs"],
                       last_epoch_macro_f1_progress_only=history[-1]["macro_f1"] if history else None)
            if not saved["completed"]:
                row["status"] = "incomplete: đang chạy hoặc bị ngắt"
                progress.append(row)
                continue
            run = load_transformer_run(folder, require_full=True)  # Hash sai/mất weights: báo lỗi.
            row["status"] = "complete: full validation đã kiểm hash"
            progress.append(row)
            full_runs[architecture].append(run)
            final_rows.append({"architecture": architecture, "seed": seed,
                               "selected_epoch": run["selected_epoch"],
                               "macro_f1_validation_fixed": run["metrics"]["macro_f1"],
                               "micro_f1_validation_fixed": run["metrics"]["micro_f1"],
                               "hamming_loss_validation_fixed": run["metrics"]["hamming_loss"]})
    return progress, final_rows, full_runs

progress_rows, completed_rows, runs_by_architecture = read_c_status(ROOT, labels)
display(pd.DataFrame(progress_rows))
print(f"Đã complete và kiểm hash: {len(completed_rows)}/9 run C.")
if completed_rows:
    display(pd.DataFrame(completed_rows))
else:
    print("Chưa có kết quả C full hoàn tất; không tạo điểm giả.")

## 5. Mean ± std và best C
Chỉ chọn khi đủ cả ba kiến trúc × ít nhất ba seed.
Mean là mức trung bình, sample std (`ddof=1`) là độ phân tán giữa các seed, không phải khoảng tin cậy.
Chọn kiến trúc theo mean Macro-F1 validation @0.5; hòa ưu tiên std nhỏ hơn rồi ít tham số hơn.
Demo dùng checkpoint seed tốt nhất theo validation của kiến trúc thắng, không phải ensemble.

In [ ]:
print("Chỉ chạy sau đủ 9 run: python -m scripts.transformers.select_best_transformer --seeds 42 123 2026")
if all(len(group) >= 3 for group in runs_by_architecture.values()):
    summaries = summarize_architectures(runs_by_architecture)
    mean_rows = []
    for summary in summaries:
        row = {"architecture": summary["architecture"], "n_seeds": summary["n_seeds"]}
        for metric in ("macro_f1", "micro_f1", "hamming_loss"):
            mean, std = summary["metrics"][metric]["mean"], summary["metrics"][metric]["std"]
            row[metric + "_mean_std"] = f"{mean:.4f} ± {std:.4f}"
        mean_rows.append(row)
    display(pd.DataFrame(mean_rows))
else:
    print("Chưa đủ ba seed cho từng kiến trúc; chỉ đọc bảng từng seed ở trên.")
    print("Không tính mean±std từ 1/9 run hoặc lấy bảng summary cũ thay bằng chứng hiện tại.")

## 6. Phần nâng cao và đánh giá cuối
- `--weighted`: BCE pos_weight = số âm/số dương, chỉ tính bằng train, lưu folder riêng.
- Threshold tuning: chọn ngưỡng trên validation của đúng checkpoint/seed, lưu cùng identity.
- Báo F1 nhãn hiếm trước/sau và false positives; không mặc định nâng cao luôn cải thiện.
- Khóa cấu hình/ngưỡng trước test bằng protocol chung của repo. Bảng cuối phải tách validation/test, smoke/full, standard/weighted và số mỗi seed.
- Đối chiếu ít nhất ba loại lỗi trên cùng ID giữa C1/C2/C3.

Nếu chỉ có một seed ablation weighting, ghi rõ 1 seed; không suy diễn thành mean±std.

## 7. Demo D và minh chứng
App load đúng C full được chọn và kiểm mapping/hash. Không có model thì báo chưa có, không dùng A thay thế hoặc trả kết quả gán sẵn.
Nhập văn bản **tiếng Anh**, hiển thị các nhãn đạt ngưỡng và bảng 28 điểm.
Điểm sigmoid chưa hiệu chuẩn. GoEmotions là Reddit nên không tự xem như đã được kiểm chứng với phản hồi khách hàng tiếng Việt.

In [ ]:
print("python app.py --device auto")
print("Mở http://127.0.0.1:7860 và chụp kết quả tính thật để đưa vào báo cáo.")
print("Đọc hướng dẫn đầy đủ: docs/EXPERIMENTS.md")